# Evaluation

## 1. imports 

In [4]:
import pandas as pd

## 2. Load data

In [5]:
import json
import glob
from pathlib import Path
import pandas as pd

# ── Load the most recent JSON file from the articles folder ───────
ARTICLES_DIR = "../data/articles"

json_files = sorted(glob.glob(f"{ARTICLES_DIR}/*.json"), key=lambda p: Path(p).stat().st_mtime)
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {ARTICLES_DIR}")

json_path = json_files[-1]  # most recently created/modified file
print(f"Loading: {json_path}")

with open(json_path) as f:
    articles = json.load(f)

ALL_COMMODITIES = ["gold", "silver", "oil", "gas"]

# Flatten metadata + headline + body into rows
rows = []
for art in articles:
    meta = art["metadata"]
    row = {
        "obs_id":           meta["obs_id"],
        "article_date":     meta["article_date"],
        "commodity":        meta["commodity"],
        "n_words_target":   meta["n_words_target"],
        "references_break": meta.get("references_break"),
        "themes":           ", ".join(meta.get("themes", [])),
        "decoys_named":     ", ".join(meta.get("decoys_named", [])),
        "model":            meta["model"],
        "headline":         art["headline"],
        "body":             art["body"],
    }

    # one-hot commodity flags, read straight from metadata["commodities"]
    for c in ALL_COMMODITIES:
        row[c] = meta["commodities"].get(c, 0)

    # per-commodity price columns, only populated for commodities that
    # were actually sampled (others stay NaN)
    for c in ALL_COMMODITIES:
        entry = meta["prices"].get(c)
        row[f"current_price_{c}"] = entry["current_price"] if entry else None
        row[f"prices_21d_{c}"]    = entry["prices_21d"]    if entry else None

    rows.append(row)

df = pd.DataFrame(rows)

# reorder: metadata cols, then one-hot flags, then per-commodity prices, then text
meta_cols   = ["obs_id", "article_date", "commodity"]
onehot_cols = ALL_COMMODITIES
price_cols  = [f"{prefix}_{c}" for c in ALL_COMMODITIES
              for prefix in ("current_price", "prices_21d")]
other_cols  = ["references_break", "themes", "decoys_named", "n_words_target", "model"]
text_cols   = ["headline", "body"]

df = df[meta_cols + onehot_cols + price_cols + other_cols + text_cols]

out_path = str(Path(json_path).with_suffix(".csv"))
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")
print()

# ── Sanity check: distribution across commodity combinations ──────
combo_counts = df.apply(
    lambda r: "+".join(c for c in ALL_COMMODITIES if r[c] == 1), axis=1
).value_counts()
print("Commodity combination distribution:")
print(combo_counts)

n_commodities = df[ALL_COMMODITIES].sum(axis=1)
print(f"\nArticles by number of commodities: {n_commodities.value_counts().sort_index().to_dict()}")

Loading: ../data/articles/articles_all_commodities_baseline_seed7_20260722_190748.json
Saved: ../data/articles/articles_all_commodities_baseline_seed7_20260722_190748.csv
Shape: (5000, 22)

Commodity combination distribution:
gold                   1178
silver                  614
gold+silver             482
gas                     433
oil                     413
gold+oil                312
gold+gas                307
gold+silver+oil         208
silver+gas              190
gold+silver+gas         187
silver+oil              170
gold+silver+oil+gas     170
oil+gas                 137
gold+oil+gas            114
silver+oil+gas           85
Name: count, dtype: int64

Articles by number of commodities: {1: 2638, 2: 1598, 3: 594, 4: 170}


## 3. Eval XGBoost

### 3.1 template_gold_silver_struct_low_seed7_final

In [6]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

ALL_COMMODITIES = ["gold", "silver", "oil", "gas"]

# Combine headline + body as input text
df["text"] = df["headline"].fillna("") + " " + df["body"].fillna("")

# TF-IDF features
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["text"])
y = df[ALL_COMMODITIES].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Multi-label XGBoost
clf = MultiOutputClassifier(XGBClassifier(
    n_estimators   = 200,
    max_depth      = 6,
    learning_rate  = 0.1,
    eval_metric    = "logloss",
    random_state   = 42,
    tree_method    = "hist",
))
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Results, per commodity
for i, c in enumerate(ALL_COMMODITIES):
    print(f"=== {c.upper()} ===")
    print(classification_report(y_test[:, i], y_pred[:, i]))

print(f"Exact match (all {len(ALL_COMMODITIES)} correct): {np.all(y_test == y_pred, axis=1).mean():.3f}")
print(f"Mean per-label accuracy: {(y_test == y_pred).mean():.3f}")

=== GOLD ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       425
           1       1.00      1.00      1.00       575

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

=== SILVER ===
              precision    recall  f1-score   support

           0       0.99      0.98      0.98       588
           1       0.97      0.99      0.98       412

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000

=== OIL ===
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       667
           1       0.98      0.95      0.97       333

    accuracy                           0.98      1000
   macro avg       0.98      0.97      0.98      1000
weighted avg       0.98      0.98 